# $0$. Download data

In [ ]:
import gdown

file_id = '1Mk5ukmVl-bThp45WF3AzBAUHNSMg8fc7'
url = f'https://drive.google.com/uc?id={file_id}'
output = 'data.zip'

gdown.download(url, output, quiet=False)

In [ ]:
!unzip -q data.zip 'data/subset_homework*' -d .

# $1.$ Viz examples

In [ ]:
import random
from pathlib import Path
import matplotlib.pyplot as plt
import cv2

def viz_random_samples(folder_path, n=3, img_extension='jpg', title_prefix="Image"):
    folder_path = Path(folder_path)
    all_image_paths = tuple(folder_path.glob(f'*.{img_extension}'))

    if not all_image_paths:
        print(f"No images found in: {folder_path}")
        return

    print(f'The folder `{folder_path.name}` contains {len(all_image_paths)} imgs')

    sample_size = min(len(all_image_paths), n)
    random_samples = random.sample(all_image_paths, sample_size)

    plt.figure(figsize=(5 * sample_size, 5))

    for i, img_path in enumerate(random_samples):
        img_bgr = cv2.imread(img_path)
        img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)
        plt.subplot(1, sample_size, i + 1)
        plt.imshow(img_rgb)
        plt.title(f'Image: {Path(img_path).name}')
        plt.axis('off')

    plt.tight_layout()
    plt.show()

In [ ]:
viz_random_samples('/content/data/subset_homework/class_id_0')

In [ ]:
viz_random_samples('/content/data/subset_homework/class_id_1')

#$2.$ Prepare the data

In [ ]:
folder_path = '/content/data/subset_homework'
folder_path = Path(folder_path)
all_image_paths = tuple(folder_path.glob(f'*/*.jpg'))

In [ ]:
imgs   = [cv2.imread(img_path, cv2.IMREAD_UNCHANGED) for img_path in all_image_paths]
labels = [int(img_path.parent.name[-1]) for img_path in all_image_paths]

In [ ]:
import time
import numpy as np


start = time.time()
pixels = np.array(imgs).reshape(len(imgs), -1) / 255.0
stop = time.time()

print('Shape:', pixels.shape)
print('Elapsed time:', stop - start)

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    np.asarray(pixels).astype('float32'),
    np.asarray(labels).astype('float32'),
    test_size=0.2,
    random_state=42
)

print(f"Training samples: {len(X_train)}")
print(f"Testing samples: {len(X_test)}")

#$3.$ Let's build a NN

In [ ]:
import tensorflow as tf
from tensorflow.keras import Model
from tensorflow.keras.layers import Dense, Input


In [ ]:
inputs = Input(shape=(pixels.shape[1], ))
outputs = Dense(1, activation="linear")(inputs)
model = Model(inputs, outputs)

model.summary()

In [ ]:
tf.keras.utils.plot_model(model, to_file="model.png", show_shapes=True, show_layer_names=True)

In [ ]:
model.compile(optimizer='adam',
              loss='mean_squared_error',
              metrics=['binary_accuracy']
              )

In [ ]:
history = model.fit(X_train, y_train, epochs=20, batch_size=32)

In [ ]:
# Plot training history
h = history.history
epochs = range(len(h['loss']))
plt.plot(epochs, h['loss'], '.-'), plt.grid(True)
plt.xlabel('epoch'), plt.ylabel('loss')

In [ ]:
# Let's also have a looks at the learnt weights
plt.plot(model.layers[1].weights[0].numpy(), '.-'), plt.grid(True)
print(model.layers[1].weights[1].numpy(), model.layers[1].bias.numpy())

# $4.$ Perfomance eval

In [ ]:
idx = 50
pred = model.predict(pixels[idx:idx+1, ...])
print(pred, labels[idx])

In [ ]:
# 1. Evaluate on training data
train_loss, train_acc = model.evaluate(X_train, y_train, verbose=0)

# 2. Evaluate on testing data
test_loss, test_acc = model.evaluate(X_test, y_test, verbose=0)

print(f" Train Accuracy: {train_acc:.4f}")
print(f" Test Accuracy:  {test_acc:.4f}")

In [ ]:
raw_probs   = model.predict(X_test).flatten()
predictions = (raw_probs > 0.5).astype(np.int8)

In [ ]:
plt.figure(figsize=(8, 8))

for cnt, idx in enumerate(np.random.randint(0, len(X_test), 4)):
    plt.subplot(2, 2, cnt + 1)

    plt.imshow(X_test[idx].reshape(imgs[0].shape), cmap='gray')

    plt.title(f"Label: {y_test[idx]}\nPred: {predictions[idx]} (with p = {raw_probs[idx]})")

    plt.axis(False)

plt.tight_layout()
plt.show()

# $5.$ Imrpoved version of the Model

In [ ]:
from tensorflow.keras.layers import Input, Dense, Dropout
from tensorflow.keras.models import Model

inputs = Input(shape=(pixels.shape[1], ))

dense1      = Dense(64, activation="relu")(inputs)
dropout1    = Dropout(0.3)(dense1)

dense2      = Dense(16, activation="relu")(dropout1)
dropout2    = Dropout(0.2)(dense2)

outputs     = Dense(1, activation="sigmoid")(dropout2)

model = Model(inputs, outputs)

In [ ]:
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import ReduceLROnPlateau, EarlyStopping, ModelCheckpoint

optimizer = Adam(learning_rate=0.001)

lr_scheduler = ReduceLROnPlateau(
    monitor='val_loss',
    factor=0.5,
    patience=5,
    verbose=1
)

checkpoint_callback = ModelCheckpoint(
    filepath='best_model.keras',
    monitor='val_accuracy',
    mode='max',
    save_best_only=True,
    verbose=1
)

early_stop = EarlyStopping(
    monitor='val_loss',
    patience=5,
    restore_best_weights=True
)

In [ ]:
model.compile(optimizer=optimizer,
              loss='binary_crossentropy',
              metrics=['accuracy'])


history = model.fit(
    X_train, y_train,
    epochs=200,
    batch_size=32,
    validation_data=(X_test, y_test),
    callbacks=[early_stop, lr_scheduler, checkpoint_callback]
)

In [ ]:
train_loss, train_acc = model.evaluate(X_train, y_train, verbose=0)
test_loss, test_acc = model.evaluate(X_test, y_test, verbose=0)

print(f" Train Accuracy: {train_acc:.4f}")
print(f" Test Accuracy:  {test_acc:.4f}")

In [ ]:
# Plot training history
h = history.history
epochs = range(len(h['loss']))
plt.plot(epochs, h['loss'], '.-'), plt.grid(True)
plt.xlabel('epoch'), plt.ylabel('loss')